In [ ]:
import os
from pathlib import Path

Path(os.getcwd()).resolve().parents[1]

In [ ]:
import json

def path_to_dict(path, base_path):
    path = Path(path)
    base_path = Path(base_path)
    d = {"parent": path.name}
    if path.is_dir():
        d["children"] = [path_to_dict(child, base_path) for child in path.iterdir()]
    return d

# Set your target directory
os.chdir("..")
os.chdir("..")
target_directory = Path(os.getcwd())
folder_structure = path_to_dict(target_directory, target_directory.parent)

with open("folder_structure.json", "w", encoding="utf-8") as f:
    json.dump(folder_structure, f, indent=4, ensure_ascii=False, separators=(",", ": "))

print("Folder structure saved with relative paths.")

Folder structure saved with relative paths.


In [ ]:
import pandas as pd
csvs = [
    "sysml/data/initializer_properties.csv",
    "sysml/data/initializer_properties.csv",
    "sysml/data/mapping_properties.csv",
    "sysml/data/mapping_property_audit.csv",
    "sysml/data/property_operation_crosswalk.csv", 
    "sysml/data/relationships.csv"
]

for c in csvs:
    print(c.split("/")[-1])
    df = pd.read_csv(c)
    print(df.info())


# JSON Analysis:

In [ ]:
from IPython.display import display

project_root = Path(os.getcwd()).resolve().parents[1]
schema_path = project_root/"development_folder"/"files"/"mbse_schema.json"
with schema_path.open("r", encoding="utf-8") as f:
    schema = json.load(f)

In [12]:
schema['database']['tables'][0].keys()

dict_keys(['name', 'description', 'columns', 'constraints'])

In [ ]:
print(f'Top Level: {schema["database"].keys()}')
for key in ["schema", "description", "dialect", "conventions"]:
    print(f'{key}: {schema["database"][key]}')
print(f'Table Keys: {schema["database"]["tables"][0].keys()}')
for table in schema["database"]["tables"]:
    print(table)

In [ ]:
tables = schema["database"]["tables"]
foreign_keys = []

for table in tables:
    for column in table.get("columns", []):

        reference = column.get("references")
        table_name = table.get("name")

        if reference:
            foreign_keys.append({
                "table": table_name,
                "column": column["name"],
                "references_table": reference["table"],
                "references_column": reference["column"],
                "on_delete": reference.get("onDelete"),
            })

df_fks = pd.DataFrame(foreign_keys)

display(df_fks)

In [ ]:
constraints = []

for table in tables:
    table_name = table.get("name")
    for i, constraint in enumerate(table.get("constraints", [])):
        constraints.append({
            "table": table_name,
            "constraint_index": i,
            "type": constraint.get("type"),
            "columns": constraint.get("columns"),
            "condition": constraint.get("condition"),
            "expression": constraint.get("expression"),
        })

df_constraints = pd.DataFrame(constraints)

display(df_constraints)

In [37]:
import sys
sys.path.append("database/prod/backend/app")

In [ ]:
from sqlalchemy.orm import DeclarativeBase, mapped_column

class User(DeclarativeBase):
    id = mapped_column()
    user_id = 10